# [Optional] Introduction to Strands Agents

This optional module introduces the Strands Agents SDK for building AI agents. Complete this if you're new to agentic AI development before proceeding to the main Developer Journey labs.

## Learning Objectives

By the end of this notebook, you will be able to:
- Create a basic agent using the Strands Agents SDK
- Add built-in and custom tools to extend agent capabilities
- Understand the agent execution loop and metrics
- Deploy an agent to Amazon Bedrock AgentCore Runtime

**Duration**: ~30 minutes

## What is Strands Agents?

[Strands Agents](https://strandsagents.com/) is an open-source SDK for building AI agents with a model-driven approach. Instead of manually orchestrating complex workflows, Strands lets the model decide when and how to use tools.

### Core Components

| Component | Description |
|-----------|-------------|
| **Model** | The LLM that powers reasoning (e.g., Claude on Bedrock) |
| **System Prompt** | Instructions defining agent behavior and personality |
| **Tools** | Functions the agent can call to take actions |

### Agent Loop Architecture

![Strands Agent Architecture](images/strands-agent-architecture.png)

The agent autonomously decides whether to respond directly or use tools based on the user's request.

---

## 1. Setup

Configure AWS credentials and model settings.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configuration
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
MODEL_ID = "global.anthropic.claude-sonnet-5"

print(f"Region: {REGION}")
print(f"Model: {MODEL_ID}")

Region: us-east-1
Model: global.anthropic.claude-sonnet-5


---

## 2. Your First Agent

Create a simple agent with just a few lines of code. The agent uses Claude on Amazon Bedrock for reasoning.

In [2]:
from strands import Agent

# Create a basic agent
agent = Agent(
    model=MODEL_ID,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

# Invoke the agent
agent("What is Amazon Bedrock in one sentence?")

Amazon Bedrock

 is a fully managed AWS service that provides access to foundation models from leading AI companies (such as Anthropic, Meta, and Amazon itself) through a single API, en

abling developers to build and scale generative AI applications without managing underlying infrastructure.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Amazon Bedrock is a fully managed AWS service that provides access to foundation models from leading AI companies (such as Anthropic, Meta, and Amazon itself) through a single API, enabling developers to build and scale generative AI applications without managing underlying infrastructure.'}], 'metadata': {'usage': {'inputTokens': 43, 'outputTokens': 91, 'totalTokens': 134}, 'metrics': {'latencyMs': 2247, 'timeToFirstByteMs': 1497}}, 'tracking_id': '1b132505-c683-aaca-730a-a55e29883816'}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[2.2600796222686768], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='50d36dfc-b5a5-395b-d858-88ab8b048f6c', usage={'inputTokens': 43, 'outputTokens': 91, 'totalTokens': 134})], usage={'inputTokens': 43, 'outputTokens': 91, 'totalTokens': 134})], traces=[<strands.telemetry.metrics.Trace object at 0xffff4d55

<div class="alert alert-block alert-info">
<b>Note:</b> The agent automatically handles the conversation with the model. You simply call the agent like a function with your input.
</div>

---

## 3. Adding Tools

Tools extend agent capabilities beyond text generation. The agent autonomously decides when to use tools based on the user's request.

### Built-in Tools

Strands provides common tools out of the box via the `strands-agents-tools` package:

In [3]:
from strands import Agent
from strands_tools import calculator, current_time

# Create agent with built-in tools
agent_with_tools = Agent(
    model=MODEL_ID,
    tools=[calculator, current_time],
    system_prompt="You are a helpful assistant with access to a calculator and clock."
)

# The agent automatically uses the calculator tool
agent_with_tools("What is 15% of 250?")

DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 250 * 0.15          │                                                                            │
│  │ Result    │ 37.5                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

15% of 250 is **37.5**.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '15% of 250 is **37.5**.'}], 'metadata': {'usage': {'inputTokens': 2931, 'outputTokens': 15, 'totalTokens': 2946}, 'metrics': {'latencyMs': 1752, 'timeToFirstByteMs': 4734}}, 'tracking_id': '2e37e8d6-541d-6c93-626c-211f53c50c11'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'example-id-df263355c0', 'name': 'calculator', 'input': {'expression': '250 * 0.15'}}, call_count=1, success_count=1, error_count=0, total_time=0.006092548370361328)}, cycle_durations=[1.9631552696228027, 4.793765544891357], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='24442bd2-6c70-42f8-8484-982ed562915a', usage={'inputTokens': 2843, 'outputTokens': 76, 'totalTokens': 2919}), EventLoopCycleMetric(event_loop_cycle_id='9e014d06-8aad-c7a2-6c3f-18fc342a0cf4', usage={'inputTokens': 2931, 'outputTokens': 15, 'totalTokens': 2946})], usage={'in

In [4]:
# The agent uses the current_time tool
agent_with_tools("What time is it now?")

DEPRECATION WARNING: current_time is deprecated. This warning becomes an error log in v0.9.0. Migration path: inject the current time as context with ContextInjector instead of calling a tool. See https://strandsagents.com/docs/user-guide/concepts/plugins/context-injector/



Tool #2: current_time


The

 current time is **07:19:03 UTC** on **October 21, 2026**.

If you'd like the time in a specific timezone, just let me know!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "The current time is **07:19:03 UTC** on **October 21, 2026**.\n\nIf you'd like the time in a specific timezone, just let me know!"}], 'metadata': {'usage': {'inputTokens': 3004, 'outputTokens': 49, 'totalTokens': 3053}, 'metrics': {'latencyMs': 3430, 'timeToFirstByteMs': 6063}}, 'tracking_id': 'e0569c9e-a36c-e04b-8c40-fff3b492ab60'}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'example-id-df263355c0', 'name': 'calculator', 'input': {'expression': '250 * 0.15'}}, call_count=1, success_count=1, error_count=0, total_time=0.006092548370361328), 'current_time': ToolMetrics(tool={'toolUseId': 'example-id-800e6b5ff9', 'name': 'current_time', 'input': {}}, call_count=1, success_count=1, error_count=0, total_time=0.0008990764617919922)}, cycle_durations=[1.9631552696228027, 4.793765544891357, 1.8420970439910889, 6.593365907669067], agent_invocations=[Age

### Custom Tools

Create domain-specific tools using the `@tool` decorator. This is essential for building agents that interact with your business systems.

In [5]:
from strands import Agent, tool

@tool
def lookup_order(order_id: str) -> str:
    """Look up order status by order ID.
    
    Args:
        order_id: The order ID to look up (e.g., ORD-12345)
    
    Returns:
        Order status information
    """
    # Simulated order database
    orders = {
        "ORD-12345": {"status": "Shipped", "eta": "2 days", "carrier": "FedEx"},
        "ORD-67890": {"status": "Processing", "eta": "5 days", "carrier": "Pending"},
    }
    
    if order_id in orders:
        order = orders[order_id]
        return f"Order {order_id}: {order['status']}, ETA: {order['eta']}, Carrier: {order['carrier']}"
    return f"Order {order_id} not found"


@tool
def check_inventory(product_sku: str) -> str:
    """Check inventory for a product SKU.
    
    Args:
        product_sku: The product SKU to check
    
    Returns:
        Inventory status
    """
    # Simulated inventory
    inventory = {
        "SKU-KB-001": {"name": "Wireless Keyboard", "qty": 150, "warehouse": "Seattle"},
        "SKU-MS-002": {"name": "Gaming Mouse", "qty": 0, "warehouse": "N/A"},
    }
    
    if product_sku in inventory:
        item = inventory[product_sku]
        status = "In Stock" if item['qty'] > 0 else "Out of Stock"
        return f"{item['name']}: {status} ({item['qty']} units in {item['warehouse']})"
    return f"Product {product_sku} not found"


print("Custom tools defined: lookup_order, check_inventory")

Custom tools defined: lookup_order, check_inventory


<div class="alert alert-block alert-warning">
<b>Key Insight:</b> The docstring in your tool function is critical - it tells the model what the tool does and when to use it. Write clear, descriptive docstrings for better tool selection.
</div>

---

## 4. Customer Support Agent Example

Let's build a customer support agent similar to what we'll use in the main Developer Journey labs.

In [6]:
support_agent = Agent(
    model=MODEL_ID,
    tools=[lookup_order, check_inventory],
    system_prompt="""You are a customer support agent for CloudCommerce.

Your responsibilities:
- Help customers check order status using the lookup_order tool
- Check product inventory using the check_inventory tool
- Be friendly, professional, and helpful

Always use the available tools to get accurate information.
If you cannot find information, apologize and offer alternatives."""
)

print("Customer support agent created with 2 tools")

Customer support agent created with 2 tools


In [7]:
# Test order lookup
support_agent("Where is my order ORD-12345?")

I'll check that for you right away.


Tool #1: lookup_order


Your order **ORD-12

345** has shipped! Here are the details:

- **Status:** Shipped 📦
- **Carrier:** FedEx
- **Estimated Delivery:

** 2 days

Is there anything else you'd like to know about this order, or can I help with anything else?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Your order **ORD-12345** has shipped! Here are the details:\n\n- **Status:** Shipped 📦\n- **Carrier:** FedEx\n- **Estimated Delivery:** 2 days\n\nIs there anything else you'd like to know about this order, or can I help with anything else?"}], 'metadata': {'usage': {'inputTokens': 840, 'outputTokens': 94, 'totalTokens': 934}, 'metrics': {'latencyMs': 2219, 'timeToFirstByteMs': 4467}}, 'tracking_id': '01749a59-90e9-c223-4e37-26e6fb4b61de'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'example-id-30a52be280', 'name': 'lookup_order', 'input': {'order_id': 'ORD-12345'}}, call_count=1, success_count=1, error_count=0, total_time=0.00032806396484375)}, cycle_durations=[1.9904773235321045, 5.234311580657959], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='5b956d91-a61e-4364-b46b-6931b8058789', usage={'inputTokens'

In [8]:
# Test inventory check
support_agent("Do you have the wireless keyboard SKU-KB-001 in stock?")

Let me check that for you.
Tool #2: check_inventory


Good

 news! The **Wireless Keyboard (SKU-KB-001)** is currently **in stock** — we have **150 units** available at our Seattle warehou

se. 🎉

Would you like help placing an order or is there anything else I can assist you with?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Good news! The **Wireless Keyboard (SKU-KB-001)** is currently **in stock** — we have **150 units** available at our Seattle warehouse. 🎉\n\nWould you like help placing an order or is there anything else I can assist you with?'}], 'metadata': {'usage': {'inputTokens': 1075, 'outputTokens': 83, 'totalTokens': 1158}, 'metrics': {'latencyMs': 2049, 'timeToFirstByteMs': 4530}}, 'tracking_id': '24ce4777-89d6-40c3-ee59-e983b7e642bc'}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'example-id-30a52be280', 'name': 'lookup_order', 'input': {'order_id': 'ORD-12345'}}, call_count=1, success_count=1, error_count=0, total_time=0.00032806396484375), 'check_inventory': ToolMetrics(tool={'toolUseId': 'example-id-567ad489b4', 'name': 'check_inventory', 'input': {'product_sku': 'SKU-KB-001'}}, call_count=1, success_count=1, error_count=0, total_time=0.00029969215

In [9]:
# Test handling unknown order
support_agent("What's the status of order ORD-99999?")


Tool #3: lookup_order


I

'm sorry, but I couldn't find any order with the ID **ORD-99999** in our system. 😔

Here are a few things you could check

:
- Double-check the order ID for any typos (it should look like "ORD-XXXXX")
- Check

 your confirmation email for the correct order number
- Let me know if you have an alternate order ID to

 try

If you continue having trouble, I'd be happy to help you look into this

 further — just let me know your account email or any other details you have, and I'll do my best to assist!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'I\'m sorry, but I couldn\'t find any order with the ID **ORD-99999** in our system. 😔\n\nHere are a few things you could check:\n- Double-check the order ID for any typos (it should look like "ORD-XXXXX")\n- Check your confirmation email for the correct order number\n- Let me know if you have an alternate order ID to try\n\nIf you continue having trouble, I\'d be happy to help you look into this further — just let me know your account email or any other details you have, and I\'ll do my best to assist!'}], 'metadata': {'usage': {'inputTokens': 1245, 'outputTokens': 161, 'totalTokens': 1406}, 'metrics': {'latencyMs': 3482, 'timeToFirstByteMs': 1597}}, 'tracking_id': '6ed25020-c431-43e9-eb4a-a8e10df16b61'}, metrics=EventLoopMetrics(cycle_count=6, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'example-id-889963baa4', 'name': 'lookup_order', 'input': {'order_id': 'ORD-99999'}}, call_count

---

## 5. Understanding Agent Execution

After running an agent, you can inspect execution metrics to understand what happened.

In [10]:
result = support_agent("Check if SKU-MS-002 is available")

# Access metrics from the result
print("\n" + "=" * 50)
print("EXECUTION METRICS")
print("=" * 50)

if hasattr(result, 'metrics'):
    metrics = result.metrics
    if hasattr(metrics, 'accumulated_usage'):
        usage = metrics.accumulated_usage
        print(f"Input tokens:  {usage.get('inputTokens', 'N/A')}")
        print(f"Output tokens: {usage.get('outputTokens', 'N/A')}")
        print(f"Total tokens:  {usage.get('totalTokens', 'N/A')}")
else:
    print("Metrics not available in this response format")


Tool #4: check_inventory


Unfortunately, the

 **Gaming Mouse (SKU-MS-002)** is currently **out of stock** — we have 0 units available at the moment. 😔



Here are some options:
- I can let you know if you'd like to be notified when it's

 back in stock (though I'd recommend checking our website for restock alerts)
- I can check if we

 have similar products available, if you let me know the SKU
- You can check back later for updated availability

Is there anything else I can help you with in the meantime?
EXECUTION METRICS
Input tokens:  8935
Output tokens: 790
Total tokens:  9725


---

## 6. Deploy to Amazon Bedrock AgentCore Runtime

Amazon Bedrock AgentCore Runtime provides a managed, serverless environment for running agents in production.

![AgentCore Runtime Architecture](images/agentcore-runtime-architecture.png)

### What is AgentCore Runtime?

AgentCore Runtime is a fully managed service that hosts your AI agents:

1. **Packages your code** into a container
2. **Creates AWS resources** (IAM roles, ECR, etc.)
3. **Hosts your agent** in a serverless environment
4. **Provides an API endpoint** for invocation

### Key Benefits

| Feature | Description |
|---------|-------------|
| **Serverless** | No infrastructure to manage |
| **Auto-scaling** | Handles variable load automatically |
| **Session Isolation** | Each conversation runs in isolated context |
| **Framework Agnostic** | Works with Strands, LangGraph, CrewAI, and more |

### Step 1: Create Agent File

Create a file called `demo_agent.py` with your agent wrapped in `BedrockAgentCoreApp`:

In [11]:
%%writefile demo_agent.py
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool

app = BedrockAgentCoreApp()

@tool
def lookup_order(order_id: str) -> str:
    """Look up order status by order ID."""
    orders = {
        "ORD-12345": {"status": "Shipped", "eta": "2 days", "carrier": "FedEx"},
        "ORD-67890": {"status": "Processing", "eta": "5 days", "carrier": "Pending"},
    }
    if order_id in orders:
        order = orders[order_id]
        return f"Order {order_id}: {order['status']}, ETA: {order['eta']}, Carrier: {order['carrier']}"
    return f"Order {order_id} not found"

agent = Agent(
    model="global.anthropic.claude-sonnet-5",
    tools=[lookup_order],
    system_prompt="You are a helpful customer support agent."
)

@app.entrypoint
def invoke(payload):
    response = agent(payload.get("prompt", ""))
    return {"result": response.message}

if __name__ == "__main__":
    app.run()

Writing demo_agent.py


In [12]:
%%writefile requirements-demo.txt
bedrock-agentcore==1.23.1
strands-agents==1.57.0


Writing requirements-demo.txt


### Step 2: Configure and Deploy

Use the AgentCore Starter Toolkit to configure and deploy the agent:

In [2]:
# Remove generated deployment files before configuring this demo.
# They are shared by the notebooks in this directory.
from pathlib import Path

for name in (".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"):
    path = Path(name)
    if path.exists():
        path.unlink()
        print(f"Removed generated deployment file: {name}")


In [14]:
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize the Runtime
agentcore_runtime = Runtime()
AGENT_NAME = "demo_agent"

# Configure the agent
agentcore_runtime.configure(
    deployment_type="container",
    disable_otel=True,
    entrypoint="demo_agent.py",
    requirements_file="requirements-demo.txt",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=REGION,
    agent_name=AGENT_NAME
)

print("Agent configured")

Entrypoint parsed: file=/workshop/03-developer-journey/demo_agent.py, bedrock_agentcore_name=demo_agent


Memory disabled - agent will be stateless


Configuring BedrockAgentCore agent: demo_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

...

📄 Generated Dockerfile: /workshop/03-developer-journey/Dockerfile

Generated .dockerignore: /workshop/03-developer-journey/.dockerignore


Setting 'demo_agent' as default agent


Bedrock AgentCore configured: /workshop/03-developer-journey/.bedrock_agentcore.yaml


Agent configured


### Step 3: Deploy and Test

Deploy the agent and test it:

In [15]:
# Deploy to AgentCore Runtime
# This builds the container, pushes to ECR, and creates the runtime
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print(f"Agent deployed: {launch_result.agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...


   • Deploy Python code directly to runtime


   • No Docker required (DEFAULT behavior)


   • Production-ready deployment


💡 Deployment options:


...

   /aws/bedrock-agentcore/runtimes/demo_agent-Ex2f895f9c-DEFAULT --log-stream-name-prefix "2026/10/21/\[runtime-logs]"


   /aws/bedrock-agentcore/runtimes/demo_agent-Ex2f895f9c-DEFAULT --log-stream-names "otel-rt-logs"


💡 Tail logs with: aws logs tail /aws/bedrock-agentcore/runtimes/demo_agent-Ex2f895f9c-DEFAULT --log-stream-name-prefix "2026/10/21/\[runtime-logs]" --follow


💡 Or view recent logs: aws logs tail /aws/bedrock-agentcore/runtimes/demo_agent-Ex2f895f9c-DEFAULT --log-stream-name-prefix "2026/10/21/\[runtime-logs]" --since 1h


Agent deployed: arn:aws:bedrock-agentcore:us-east-1:000000000000:runtime/demo_agent-Ex2f895f9c


In [16]:
import time

# Wait for agent to be ready
status = agentcore_runtime.status()
agent_status = status.endpoint["status"]
terminal_states = {"READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"}

while agent_status not in terminal_states:
    print(f"Agent status: {agent_status}")
    time.sleep(10)
    status = agentcore_runtime.status()
    agent_status = status.endpoint["status"]

print(f"Final status: {agent_status}")

Retrieved Bedrock AgentCore status for: demo_agent


Final status: READY


In [17]:
import os
import json
import uuid
import boto3

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
data_client = boto3.client("bedrock-agentcore", region_name=region)

# Get agent ARN from deployment result or runtime status
try:
    agent_arn = launch_result.agent_arn
except NameError:
    status = agentcore_runtime.status()
    agent_arn = status.runtime["agentRuntimeArn"]


def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    global LAST_SESSION_ID
    LAST_SESSION_ID = str(uuid.uuid4())
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=LAST_SESSION_ID,
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))


# Test the deployed agent
print(f"Agent ARN: {agent_arn}")
result = invoke_agent("where is my order ORD-12345?")
print(result)

# Assert the runtime is serving demo_agent.py, not another lab's agent.
# demo_agent.py returns {"result": ...}; a stale routing agent returns
# {"response", "model_used", "complexity"}. Fail loudly on mismatch.
assert "result" in result, (
    f"Deployed agent returned unexpected shape {list(result.keys())} - "
    "the runtime is serving a different agent than demo_agent.py. "
    "Re-run the guard cell to clear stale config, then redeploy."
)
assert "ORD-12345" in str(result["result"]), (
    "Agent did not return order ORD-12345 - the lookup_order tool is missing. "
    "The wrong container is deployed; clear stale config and redeploy."
)
print("\n✅ Verified: runtime is serving demo_agent.py with the lookup_order tool")

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:000000000000:runtime/demo_agent-Ex2f895f9c


{'result': {'role': 'assistant', 'content': [{'text': "Your order **ORD-12345** has shipped and is on its way via **FedEx**! 📦\n\n**Estimated delivery:** 2 days\n\nIf you'd like, I can help you find a tracking link or answer any other questions about your order."}], 'metadata': {'usage': {'inputTokens': 581, 'outputTokens': 76, 'totalTokens': 657}, 'metrics': {'latencyMs': 2231, 'timeToFirstByteMs': 1513}}, 'tracking_id': 'b7f37e23-58fb-ec34-d98b-1140b7851fef'}}

✅ Verified: runtime is serving demo_agent.py with the lookup_order tool


In [18]:
# Clean up AgentCore resources using Python API
agentcore_runtime.destroy(delete_ecr_repo=True)

# Clean up local files
import os
for f in ['demo_agent.py', 'requirements-demo.txt', '.bedrock_agentcore.yaml', 'Dockerfile', '.dockerignore']:
    if os.path.exists(f):
        os.remove(f)

print("Cleanup complete!")

🗑️ Destroying Bedrock AgentCore resources


   • Including ECR repository deletion


Starting destroy operation for agent: demo_agent (dry_run=False, delete_ecr_repo=True)


DEFAULT endpoint will be automatically deleted with agent


Deleted AgentCore agent: arn:aws:bedrock-agentcore:us-east-1:000000000000:runtime/demo_agent-Ex2f895f9c


...

Cleared default agent (no agents remaining)


Removed configuration file: /workshop/03-developer-journey/.bedrock_agentcore.yaml


Destroy operation completed. Resources removed: 10, Warnings: 0, Errors: 0


Destroy completed. Removed 10 resources


Cleanup complete!


---

## Summary

| Concept | Description |
|---------|-------------|
| **Agent** | Model + System Prompt + Tools |
| **Tools** | Functions the agent can call |
| **@tool decorator** | Creates custom tools from Python functions |
| **Agent Loop** | Reason -> Select Tool -> Execute -> Repeat/Respond |
| **AgentCore Runtime** | Managed serverless environment for production |

### Key Takeaways

1. **Simple API**: Create agents with just 3 components (model, prompt, tools)
2. **Tool-driven**: The model decides when and how to use tools
3. **Custom tools**: Use `@tool` decorator for domain-specific functionality
4. **Production-ready**: AgentCore Runtime handles scaling and infrastructure

---

> **Note:** The tools used in this introduction (`lookup_order`, `check_inventory`) are different from the tools used in the main Developer Journey labs (`get_return_policy`, `get_product_info`, `get_technical_support`). The concepts you learned here — creating agents, defining custom tools, and deploying to AgentCore — apply the same way throughout the workshop.

**Next**: Continue to the main Developer Journey labs to build a complete customer support system with observability and evaluation.